# A3 — full shader set: phong, texture, bump, displacement

In [1]:
import numpy as np

def blinn_phong(payload, lights, eye_pos, kd=None):
    ka = np.array([0.005, 0.005, 0.005])
    if kd is None:
        kd = payload['color']
    ks = np.array([0.7937, 0.7937, 0.7937])
    p = 150.0
    amb_intensity = np.array([10, 10, 10])
    n = payload['normal'] / np.linalg.norm(payload['normal'])
    point = payload['pos']
    v = (eye_pos - point) / np.linalg.norm(eye_pos - point)
    result = ka * amb_intensity
    for L in lights:
        l_vec = L['pos'] - point
        r2 = float(l_vec @ l_vec)
        l = l_vec / np.sqrt(r2)
        h = (l + v) / np.linalg.norm(l + v)
        diff = kd * (L['I'] / r2) * max(0.0, float(n @ l))
        spec = ks * (L['I'] / r2) * (max(0.0, float(n @ h)) ** p)
        result += diff + spec
    return np.clip(result * 255, 0, 255)


## Texture shader

In [2]:
def sample_nearest(tex, u, v):
    h, w = tex.shape[:2]
    u = min(max(u, 0.0), 1.0)
    v = min(max(v, 0.0), 1.0)
    return tex[int((1 - v) * (h - 1)), int(u * (w - 1))]

def texture_shader(payload, tex, lights, eye_pos):
    u, v = payload['uv']
    kd = sample_nearest(tex, u, v) / 255.0
    return blinn_phong(payload, lights, eye_pos, kd=kd)


## Bilinear sampling (bonus)

In [3]:
def sample_bilinear(tex, u, v):
    h, w = tex.shape[:2]
    x = u * (w - 1)
    y = (1 - v) * (h - 1)
    x0, y0 = int(np.floor(x)), int(np.floor(y))
    x1, y1 = min(x0 + 1, w - 1), min(y0 + 1, h - 1)
    fx, fy = x - x0, y - y0
    c00 = tex[y0, x0]; c10 = tex[y0, x1]
    c01 = tex[y1, x0]; c11 = tex[y1, x1]
    return (c00*(1-fx)*(1-fy) + c10*fx*(1-fy)
            + c01*(1-fx)*fy + c11*fx*fy)


## Bump + displacement

In [4]:
def bump_shader(payload, height_map):
    n = payload['normal'] / np.linalg.norm(payload['normal'])
    # local frame -- t and b picked to match GAMES101 setup
    x, y, z = n
    t = np.array([x*y/np.sqrt(x*x + z*z), np.sqrt(x*x + z*z), z*y/np.sqrt(x*x + z*z)])
    b = np.cross(n, t)
    TBN = np.column_stack([t, b, n])
    u, v = payload['uv']
    h, w = height_map.shape[:2]
    kh, kn = 0.2, 0.1
    def H(uu, vv):
        return float(sample_nearest(height_map, uu, vv).mean()) / 255.0
    dU = kh * kn * (H(u + 1.0/w, v) - H(u, v))
    dV = kh * kn * (H(u, v + 1.0/h) - H(u, v))
    ln = np.array([-dU, -dV, 1.0])
    return (TBN @ ln) / np.linalg.norm(TBN @ ln)

def displacement_shader(payload, height_map, lights, eye_pos):
    n_new = bump_shader(payload, height_map)
    u, v = payload['uv']
    kh, kn = 0.2, 0.1
    h_val = float(sample_nearest(height_map, u, v).mean()) / 255.0
    payload = dict(payload)
    payload['pos'] = payload['pos'] + kn * payload['normal'] * h_val
    payload['normal'] = n_new
    return blinn_phong(payload, lights, eye_pos)
